# THREAD — H&M Fashion Recommendation Lab

Popularity → personal candidates → learned scoring → chronological evaluation.

**Before running:** accept the competition rules yourself, then attach the
official **H&M Personalized Fashion Recommendations** competition through
**Add Input**. CPU is sufficient; Internet can stay off. No image archive is read.

This notebook creates local outputs only. It does not publish anything or submit
predictions. No real-data result is claimed until this notebook finishes.


In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path('/kaggle/working/thread_source')
package = ROOT / 'fashion_recs'
package.mkdir(parents=True, exist_ok=True)
SOURCES = json.loads('{"__init__.py": "\\"\\"\\"Fashion recommendations with chronological evaluation.\\"\\"\\"\\n\\n", "data.py": "from pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\n\\n\\ndef load_data(directory: Path, lookback_days: int = 98):\\n    \\"\\"\\"Read only the recent window, without loading all 31M rows at once.\\"\\"\\"\\n    path = directory / \\"transactions_train.csv\\"\\n    if not path.exists():\\n        raise FileNotFoundError(f\\"Missing {path}. See README.md for Kaggle access.\\")\\n    latest = None\\n    total = 0\\n    for chunk in pd.read_csv(path, usecols=[\\"t_dat\\"], chunksize=1_000_000):\\n        maximum = pd.to_datetime(chunk.t_dat).max()\\n        latest = maximum if latest is None else max(latest, maximum)\\n        total += len(chunk)\\n    if latest is None or pd.isna(latest):\\n        raise ValueError(\\"No transactions found\\")\\n    end = latest.normalize() + pd.Timedelta(days=1)\\n    start = end - pd.Timedelta(days=lookback_days)\\n    recent = []\\n    for chunk in pd.read_csv(\\n        path, usecols=[\\"t_dat\\", \\"customer_id\\", \\"article_id\\"],\\n        dtype={\\"customer_id\\": \\"string\\", \\"article_id\\": \\"string\\"}, chunksize=1_000_000,\\n    ):\\n        chunk[\\"t_dat\\"] = pd.to_datetime(chunk.t_dat)\\n        recent.append(chunk.loc[chunk.t_dat.ge(start)].copy())\\n    transactions = pd.concat(recent, ignore_index=True)\\n    transactions[\\"article_id\\"] = transactions.article_id.str.zfill(10)\\n    articles = pd.read_csv(directory / \\"articles.csv\\", dtype={\\"article_id\\": \\"string\\"})\\n    articles[\\"article_id\\"] = articles.article_id.str.zfill(10)\\n    return transactions, articles, end, {\\n        \\"source_rows\\": total, \\"loaded_rows\\": len(transactions),\\n        \\"lookback_days\\": lookback_days, \\"source_max_date\\": str(latest.date()),\\n    }\\n\\n\\ndef synthetic_data(seed: int = 42):\\n    \\"\\"\\"Small, artificial shopping histories; never presented as H&M results.\\"\\"\\"\\n    rng = np.random.default_rng(seed)\\n    categories = [\\"Knitwear\\", \\"Trousers\\", \\"Shirts\\", \\"Dresses\\", \\"Accessories\\", \\"Shoes\\"]\\n    palette = [\\"Black\\", \\"Cream\\", \\"Blue\\", \\"Olive\\", \\"Burgundy\\", \\"Sand\\"]\\n    articles = pd.DataFrame([\\n        {\\"article_id\\": f\\"{i + 1:010d}\\", \\"prod_name\\": f\\"{categories[i % 6]} {i // 6 + 1:02d}\\",\\n         \\"product_type_name\\": categories[i % 6], \\"colour_group_name\\": palette[(i // 6) % 6],\\n         \\"product_group_name\\": categories[i % 6]}\\n        for i in range(180)\\n    ])\\n    end = pd.Timestamp(\\"2020-09-23\\")\\n    rows = []\\n    for user in range(260):\\n        favourite = user % 6\\n        pool = np.arange(favourite, 180, 6)\\n        personal = rng.choice(pool, size=7, replace=False)\\n        for day in range(91):\\n            if rng.random() > 0.22:\\n                continue\\n            for _ in range(int(rng.integers(1, 4))):\\n                pick = int(rng.choice(personal)) if rng.random() < 0.7 else int(rng.choice(pool))\\n                if rng.random() < 0.08:\\n                    pick = int(rng.integers(0, 180))\\n                rows.append((end - pd.Timedelta(days=91 - day), f\\"demo-{user:04d}\\", f\\"{pick + 1:010d}\\"))\\n    return pd.DataFrame(rows, columns=[\\"t_dat\\", \\"customer_id\\", \\"article_id\\"]), articles, end\\n\\n", "metrics.py": "from collections.abc import Iterable\\n\\n\\ndef apk(actual: Iterable[str], predicted: Iterable[str], k: int = 12) -> float:\\n    \\"\\"\\"AP@k; duplicate recommendations never earn additional credit.\\"\\"\\"\\n    truth = set(actual)\\n    if not truth or k <= 0:\\n        return 0.0\\n    hits, score, seen = 0, 0.0, set()\\n    for rank, item in enumerate(list(predicted)[:k], 1):\\n        if item in truth and item not in seen:\\n            hits += 1\\n            score += hits / rank\\n        seen.add(item)\\n    return score / min(len(truth), k)\\n\\n", "model.py": "from collections import Counter, defaultdict\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.ensemble import HistGradientBoostingClassifier\\n\\nFEATURES = [\\"popularity_7d\\", \\"popularity_28d\\", \\"repeat_recency\\", \\"repeat_count\\", \\"category_affinity\\", \\"transition_score\\"]\\n\\n\\nclass CandidateEngine:\\n    \\"\\"\\"Recent popularity + repeat purchases + consecutive-purchase neighbours.\\n\\n    Every statistic uses dates strictly before as_of. The neighbour graph is\\n    a lightweight item-to-item signal, not a causal claim about a shopping basket.\\n    \\"\\"\\"\\n\\n    def __init__(self, popular_candidates=60, neighbours=12, history_items=12):\\n        self.popular_candidates = popular_candidates\\n        self.neighbours = neighbours\\n        self.history_items = history_items\\n\\n    def fit(self, transactions, articles, as_of):\\n        self.as_of = pd.Timestamp(as_of)\\n        frame = transactions.loc[transactions.t_dat.lt(self.as_of)].copy()\\n        if frame.empty:\\n            raise ValueError(\\"No history strictly before cutoff\\")\\n        self.max_seen_date = frame.t_dat.max()\\n        self.categories = articles.set_index(\\"article_id\\").product_type_name.fillna(\\"Unknown\\").to_dict()\\n        self.count7 = frame.loc[frame.t_dat.ge(self.as_of - pd.Timedelta(days=7)), \\"article_id\\"].value_counts().to_dict()\\n        self.count28 = frame.loc[frame.t_dat.ge(self.as_of - pd.Timedelta(days=28)), \\"article_id\\"].value_counts().to_dict()\\n        counts = frame.article_id.value_counts().to_dict()\\n        self.popular = sorted(counts, key=lambda a: (-self.count7.get(a, 0), -self.count28.get(a, 0), -counts[a], a))\\n        personal = frame.groupby([\\"customer_id\\", \\"article_id\\"], observed=True).agg(\\n            last=(\\"t_dat\\", \\"max\\"), count=(\\"article_id\\", \\"size\\")\\n        ).reset_index().sort_values([\\"customer_id\\", \\"last\\", \\"article_id\\"], ascending=[True, False, True])\\n        personal = personal.groupby(\\"customer_id\\", sort=False).head(self.history_items)\\n        self.histories = defaultdict(list)\\n        self.affinities = defaultdict(Counter)\\n        for row in personal.itertuples(index=False):\\n            age = (self.as_of - row.last).days\\n            self.histories[row.customer_id].append((row.article_id, age, int(row.count)))\\n            self.affinities[row.customer_id][self.categories.get(row.article_id, \\"Unknown\\")] += int(row.count)\\n        # Bound graph construction: at most six recent events per customer.\\n        graph_frame = frame.loc[frame.t_dat.ge(self.as_of - pd.Timedelta(days=28))].sort_values(\\n            [\\"customer_id\\", \\"t_dat\\", \\"article_id\\"], kind=\\"stable\\"\\n        ).groupby(\\"customer_id\\", sort=False).tail(6).copy()\\n        graph_frame[\\"previous\\"] = graph_frame.groupby(\\"customer_id\\").article_id.shift()\\n        pairs = graph_frame.loc[graph_frame.previous.notna() & graph_frame.previous.ne(graph_frame.article_id)]\\n        forward = pairs[[\\"previous\\", \\"article_id\\"]].rename(columns={\\"previous\\": \\"source\\", \\"article_id\\": \\"target\\"})\\n        reverse = forward.rename(columns={\\"source\\": \\"target\\", \\"target\\": \\"source\\"})\\n        edges = pd.concat([forward, reverse]).groupby([\\"source\\", \\"target\\"]).size().rename(\\"count\\").reset_index()\\n        edges = edges.sort_values([\\"source\\", \\"count\\", \\"target\\"], ascending=[True, False, True])\\n        edges = edges.groupby(\\"source\\", sort=False).head(self.neighbours)\\n        self.graph = defaultdict(list)\\n        for row in edges.itertuples(index=False):\\n            self.graph[row.source].append((row.target, int(row.count)))\\n        return self\\n\\n    def candidates(self, user):\\n        history = self.histories.get(user, [])\\n        personal = {article: (age, count) for article, age, count in history}\\n        neighbours = Counter()\\n        for article, age, _ in history[:6]:\\n            for neighbour, count in self.graph.get(article, []):\\n                neighbours[neighbour] += np.exp(-age / 28) * count\\n        ids = list(dict.fromkeys(self.popular[:self.popular_candidates] + list(personal) + list(neighbours)))\\n        affinity = self.affinities.get(user, {})\\n        denominator = max(sum(affinity.values()), 1)\\n        matrix = np.array([\\n            [np.log1p(self.count7.get(a, 0)), np.log1p(self.count28.get(a, 0)),\\n             np.exp(-personal[a][0] / 28) if a in personal else 0,\\n             np.log1p(personal[a][1]) if a in personal else 0,\\n             affinity.get(self.categories.get(a, \\"Unknown\\"), 0) / denominator,\\n             np.log1p(neighbours.get(a, 0))]\\n            for a in ids\\n        ], dtype=np.float32)\\n        return ids, matrix\\n\\n    def recommend(self, user, method=\\"hybrid\\", ranker=None, k=12):\\n        if method == \\"popular\\":\\n            return self.popular[:k]\\n        if method not in {\\"hybrid\\", \\"learned\\"}:\\n            raise ValueError(f\\"Unknown method: {method}\\")\\n        ids, features = self.candidates(user)\\n        # A cold customer has no personal signal; popularity is an explicit fallback.\\n        if not self.histories.get(user):\\n            return self.popular[:k]\\n        if method == \\"learned\\":\\n            if ranker is None:\\n                raise ValueError(\\"A fitted ranker is required\\")\\n            scores = ranker.predict_proba(features)[:, 1]\\n        else:\\n            scores = features @ np.array([0.35, 0.1, 3.0, 0.7, 1.0, 0.3])\\n        order = np.argsort(-scores, kind=\\"stable\\")[:k]\\n        return [ids[i] for i in order]\\n\\n\\ndef fit_ranker(engine, labels, seed=42):\\n    rows, targets = [], []\\n    for user, actual in labels.items():\\n        ids, matrix = engine.candidates(user)\\n        rows.append(matrix)\\n        truth = set(actual)\\n        targets.extend(int(item in truth) for item in ids)\\n    x = np.concatenate(rows)\\n    y = np.asarray(targets)\\n    if len(np.unique(y)) < 2:\\n        raise ValueError(\\"Training candidates contain no positive examples; increase training users\\")\\n    # Fixed iteration count; sklearn\'s default random validation split is disabled.\\n    ranker = HistGradientBoostingClassifier(\\n        max_iter=70, max_leaf_nodes=15, max_depth=5, min_samples_leaf=40,\\n        learning_rate=0.08, l2_regularization=3.0, early_stopping=False, random_state=seed,\\n    )\\n    ranker.fit(x, y, sample_weight=np.where(y == 1, 15.0, 1.0))\\n    return ranker, {\\"rows\\": len(y), \\"positives\\": int(y.sum()), \\"features\\": FEATURES}\\n\\n", "pipeline.py": "import argparse\\nimport hashlib\\nimport json\\nimport pickle\\nimport platform\\nimport time\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nimport sklearn\\n\\nfrom .data import load_data, synthetic_data\\nfrom .metrics import apk\\nfrom .model import CandidateEngine, fit_ranker\\n\\n\\ndef week_labels(transactions, start, max_users, seed):\\n    frame = transactions.loc[transactions.t_dat.ge(start) & transactions.t_dat.lt(start + pd.Timedelta(days=7))]\\n    labels = frame.groupby(\\"customer_id\\").article_id.agg(lambda s: list(dict.fromkeys(s))).to_dict()\\n    users = sorted(labels)\\n    if max_users and len(users) > max_users:\\n        selected = np.random.default_rng(seed).choice(users, size=max_users, replace=False)\\n        labels = {user: labels[user] for user in sorted(selected)}\\n    return labels, int(frame.customer_id.nunique())\\n\\n\\ndef bootstrap_difference(candidate, baseline, seed=42):\\n    diff = np.asarray(candidate) - np.asarray(baseline)\\n    rng = np.random.default_rng(seed)\\n    means = [float(rng.choice(diff, size=len(diff), replace=True).mean()) for _ in range(500)]\\n    return [float(x) for x in np.quantile(means, [0.025, 0.975])]\\n\\n\\ndef evaluate(engine, ranker, labels):\\n    per_user = {name: [] for name in [\\"popular\\", \\"hybrid\\", \\"learned\\"]}\\n    recalls, catalogue = [], set()\\n    cold_count, cold_scores, warm_scores = 0, [], []\\n    for user, actual in labels.items():\\n        candidates, _ = engine.candidates(user)\\n        recalls.append(len(set(actual) & set(candidates)) / len(set(actual)))\\n        for name in per_user:\\n            recommendations = engine.recommend(user, name, ranker)\\n            score = apk(actual, recommendations)\\n            per_user[name].append(score)\\n            if name == \\"learned\\":\\n                catalogue.update(recommendations)\\n                if user not in engine.histories:\\n                    cold_count += 1\\n                    cold_scores.append(score)\\n                else:\\n                    warm_scores.append(score)\\n    metrics = {name: float(np.mean(scores)) for name, scores in per_user.items()}\\n    return {\\n        \\"map_at_12\\": metrics,\\n        \\"learned_minus_popular_95pct_bootstrap_ci\\": bootstrap_difference(per_user[\\"learned\\"], per_user[\\"popular\\"]),\\n        \\"candidate_recall\\": float(np.mean(recalls)),\\n        \\"catalogue_coverage\\": len(catalogue) / len(engine.popular),\\n        \\"evaluated_users\\": len(labels), \\"cold_users\\": cold_count,\\n        \\"cold_map_at_12\\": float(np.mean(cold_scores)) if cold_scores else None,\\n        \\"warm_map_at_12\\": float(np.mean(warm_scores)) if warm_scores else None,\\n    }\\n\\n\\ndef run(transactions, articles, end, output, *, synthetic, train_users=3000, eval_users=5000, seed=42, source=None):\\n    started = time.perf_counter()\\n    output.mkdir(parents=True, exist_ok=True)\\n    training_start = end - pd.Timedelta(days=14)\\n    evaluation_start = end - pd.Timedelta(days=7)\\n    training_labels, total_train = week_labels(transactions, training_start, train_users, seed)\\n    evaluation_labels, total_eval = week_labels(transactions, evaluation_start, eval_users, seed)\\n    if not training_labels or not evaluation_labels:\\n        raise ValueError(\\"At least two complete, non-empty label weeks are required\\")\\n    print(f\\"Training labels: {len(training_labels):,}; evaluation: {len(evaluation_labels):,}\\", flush=True)\\n    training_engine = CandidateEngine().fit(transactions, articles, training_start)\\n    training_feature_max_date = str(training_engine.max_seen_date.date())\\n    ranker, training_info = fit_ranker(training_engine, training_labels, seed)\\n    del training_engine\\n    evaluation_engine = CandidateEngine().fit(transactions, articles, evaluation_start)\\n    metrics = evaluate(evaluation_engine, ranker, evaluation_labels)\\n    report = {\\n        \\"data_kind\\": \\"SYNTHETIC DEMO \\u2014 NOT H&M RESULTS\\" if synthetic else \\"H&M competition data\\",\\n        \\"seed\\": seed, \\"k\\": 12, \\"training_labels_start\\": str(training_start.date()),\\n        \\"evaluation_labels_start\\": str(evaluation_start.date()), \\"evaluation_labels_end_exclusive\\": str(end.date()),\\n        \\"evaluation_feature_max_date\\": str(evaluation_engine.max_seen_date.date()),\\n        \\"training_feature_max_date\\": training_feature_max_date,\\n        \\"runtime\\": {\\"python\\": platform.python_version(), \\"numpy\\": np.__version__,\\n                    \\"pandas\\": pd.__version__, \\"scikit_learn\\": sklearn.__version__},\\n        \\"training_population\\": total_train, \\"evaluation_population\\": total_eval,\\n        \\"sampling\\": \\"Seeded uniform sample of customers purchasing in each label week; includes cold customers. Zero means all.\\",\\n        \\"ranking_training\\": training_info, \\"source\\": source or {}, **metrics,\\n        \\"hyperparameter_policy\\": \\"Fixed parameters before holdout evaluation; no holdout tuning or model selection.\\",\\n    }\\n    del evaluation_engine\\n    # Fit serving features on all observed history, keeping the learned ranker fixed.\\n    final_engine = CandidateEngine().fit(transactions, articles, end)\\n    article_rows = articles.fillna(\\"\\").to_dict(orient=\\"records\\")\\n    metadata = {row[\\"article_id\\"]: row for row in article_rows}\\n    demo_users = list(evaluation_labels)[:100]\\n    payload = {\\n        \\"data_kind\\": report[\\"data_kind\\"], \\"as_of\\": str(end.date()), \\"metrics\\": metrics,\\n        \\"users\\": [], \\"popular\\": final_engine.popular[:12],\\n    }\\n    visible_ids = set(final_engine.popular[:12])\\n    for index, user in enumerate(demo_users, 1):\\n        recs = final_engine.recommend(user, \\"learned\\", ranker)\\n        history = [a for a, _, _ in final_engine.histories.get(user, [])]\\n        candidates, features = final_engine.candidates(user)\\n        feature_lookup = dict(zip(candidates, features))\\n        reasons = {}\\n        for article in recs:\\n            f = feature_lookup[article]\\n            reasons[article] = (\\n                \\"\\u041f\\u043e\\u043a\\u0443\\u043f\\u0430\\u043b\\u0438 \\u0440\\u0430\\u043d\\u044c\\u0448\\u0435\\" if f[2] > 0 else\\n                \\"\\u041f\\u043e\\u0445\\u043e\\u0436\\u0430\\u044f \\u043a\\u0430\\u0442\\u0435\\u0433\\u043e\\u0440\\u0438\\u044f \\u043f\\u043e\\u043a\\u0443\\u043f\\u043e\\u043a\\" if f[4] > 0 else\\n                \\"\\u0421\\u0432\\u044f\\u0437\\u044c \\u0441 \\u043f\\u0440\\u043e\\u0448\\u043b\\u044b\\u043c\\u0438 \\u043f\\u043e\\u043a\\u0443\\u043f\\u043a\\u0430\\u043c\\u0438\\" if f[5] > 0 else \\"\\u041f\\u043e\\u043f\\u0443\\u043b\\u044f\\u0440\\u043d\\u043e \\u0441\\u0435\\u0439\\u0447\\u0430\\u0441\\"\\n            )\\n        # Export opaque demo labels rather than actual Kaggle customer identifiers.\\n        payload[\\"users\\"].append({\\"id\\": f\\"\\u041f\\u043e\\u043a\\u0443\\u043f\\u0430\\u0442\\u0435\\u043b\\u044c {index:03d}\\", \\"history\\": history, \\"recommendations\\": recs, \\"reasons\\": reasons})\\n        visible_ids.update(recs + history)\\n    payload[\\"articles\\"] = {a: metadata.get(a, {\\"article_id\\": a}) for a in sorted(visible_ids)}\\n    report[\\"elapsed_seconds\\"] = round(time.perf_counter() - started, 2)\\n    (output / \\"report.json\\").write_text(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False), encoding=\\"utf-8\\")\\n    (output / \\"demo.json\\").write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str, allow_nan=False), encoding=\\"utf-8\\")\\n    with (output / \\"model.pkl\\").open(\\"wb\\") as stream:\\n        pickle.dump({\\"engine\\": final_engine, \\"ranker\\": ranker, \\"synthetic\\": synthetic}, stream)\\n    kind = report[\\"data_kind\\"]\\n    scores = metrics[\\"map_at_12\\"]\\n    text = f\\"\\"\\"# Evaluation report\\\\n\\\\nData: **{kind}**\\\\n\\\\nTraining labels: {training_start.date()} to {evaluation_start.date()} (exclusive).\\\\nHeld-out labels: {evaluation_start.date()} to {end.date()} (exclusive).\\\\nAll evaluation features predate {evaluation_start.date()}.\\\\n\\\\n| Model | MAP@12 |\\\\n|---|---:|\\\\n| Recent popularity | {scores[\'popular\']:.6f} |\\\\n| Fixed hybrid | {scores[\'hybrid\']:.6f} |\\\\n| Learned candidate scorer | {scores[\'learned\']:.6f} |\\\\n\\\\nEvaluated customers: {len(evaluation_labels)} / {total_eval}.\\\\nCandidate recall: {metrics[\'candidate_recall\']:.4f}.\\\\nCold customers: {metrics[\'cold_users\']}.\\\\n\\\\nSee report.json for the paired bootstrap interval and complete configuration.\\\\nThese are offline results among purchasers in one held-out week, not online business uplift.\\\\nCandidates come only from prior purchases: genuinely new catalogue items need a separate content candidate generator.\\\\nReason badges describe available signals, not causal model explanations.\\\\nNo official Kaggle score is claimed.\\\\n\\"\\"\\"\\n    (output / \\"report.md\\").write_text(text, encoding=\\"utf-8\\")\\n    hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(output.iterdir()) if p.is_file() and p.name != \\"manifest.json\\"}\\n    (output / \\"manifest.json\\").write_text(json.dumps(hashes, indent=2), encoding=\\"utf-8\\")\\n    print(json.dumps({\\"data_kind\\": kind, **metrics, \\"output\\": str(output)}, ensure_ascii=False, indent=2), flush=True)\\n    return report\\n\\n\\ndef submit(model_path, sample_path, output):\\n    # Only load locally produced pickle files. Never load an untrusted pickle.\\n    with model_path.open(\\"rb\\") as stream:\\n        model = pickle.load(stream)\\n    if model[\\"synthetic\\"]:\\n        raise ValueError(\\"Synthetic models cannot generate a competition submission\\")\\n    engine, ranker = model[\\"engine\\"], model[\\"ranker\\"]\\n    if len(engine.popular) < 12:\\n        raise ValueError(\\"At least 12 observed articles are required\\")\\n    output.parent.mkdir(parents=True, exist_ok=True)\\n    import csv\\n    with output.open(\\"w\\", newline=\\"\\", encoding=\\"utf-8\\") as stream:\\n        writer = csv.writer(stream)\\n        writer.writerow([\\"customer_id\\", \\"prediction\\"])\\n        for chunk in pd.read_csv(sample_path, usecols=[\\"customer_id\\"], dtype=\\"string\\", chunksize=10000):\\n            for user in chunk.customer_id:\\n                recommendations = engine.recommend(user, \\"learned\\", ranker)\\n                if len(recommendations) != 12 or len(set(recommendations)) != 12:\\n                    raise ValueError(\\"Submission requires 12 unique articles per customer\\")\\n                writer.writerow([user, \\" \\".join(recommendations)])\\n    print(f\\"Created {output}; not uploaded to Kaggle\\", flush=True)\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=\\"Train and evaluate a fashion recommender\\")\\n    commands = parser.add_subparsers(dest=\\"command\\", required=True)\\n    for name in [\\"demo\\", \\"train\\"]:\\n        command = commands.add_parser(name)\\n        command.add_argument(\\"--output\\", type=Path, default=Path(\\"artifacts\\") / name)\\n        command.add_argument(\\"--train-users\\", type=int, default=3000)\\n        command.add_argument(\\"--eval-users\\", type=int, default=5000)\\n        command.add_argument(\\"--seed\\", type=int, default=42)\\n        if name == \\"train\\":\\n            command.add_argument(\\"--data-dir\\", type=Path, required=True)\\n    command = commands.add_parser(\\"submit\\")\\n    command.add_argument(\\"--model\\", type=Path, required=True)\\n    command.add_argument(\\"--sample\\", type=Path, required=True)\\n    command.add_argument(\\"--output\\", type=Path, default=Path(\\"artifacts/submission.csv\\"))\\n    args = parser.parse_args()\\n    if args.command == \\"submit\\":\\n        submit(args.model, args.sample, args.output)\\n        return\\n    if args.train_users < 0 or args.eval_users < 0:\\n        parser.error(\\"User limits must be non-negative; zero means all\\")\\n    if args.command == \\"demo\\":\\n        tx, articles, end = synthetic_data(args.seed)\\n        source = {\\"generator\\": \\"fashion_recs.data.synthetic_data\\", \\"rows\\": len(tx)}\\n    else:\\n        tx, articles, end, source = load_data(args.data_dir)\\n    run(tx, articles, end, args.output, synthetic=args.command == \\"demo\\", train_users=args.train_users,\\n        eval_users=args.eval_users, seed=args.seed, source=source)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"}')
for name, content in SOURCES.items():
    (package / name).write_text(content, encoding='utf-8')
sys.path.insert(0, str(ROOT))


## Configuration

An initial reproducible run uses sampled customers to keep CPU cost reasonable.
The final observed week is reserved for evaluation. All methods use the same
evaluation customers. Set EVAL_USERS=0 to evaluate the full week; this costs more.


In [ ]:
TRAIN_USERS = 3000
EVAL_USERS = 5000
SEED = 42
OUTPUT = Path('/kaggle/working/thread_artifacts')

input_root = Path('/kaggle/input')
# Avoid recursively walking the 100k-file image tree on Kaggle's remote mount.
candidates = list(input_root.glob('*/transactions_train.csv'))
candidates += list(input_root.glob('competitions/*/transactions_train.csv'))
candidates += list(input_root.glob('datasets/*/*/transactions_train.csv'))
candidates = sorted(set(candidates))
if len(candidates) != 1:
    raise RuntimeError(f'Expected one official H&M transactions file; found {len(candidates)}. Attach competition data.')
DATA_DIR = candidates[0].parent
print('Data:', DATA_DIR, flush=True)


In [ ]:
from fashion_recs.data import load_data
from fashion_recs.pipeline import run

transactions, articles, end, source = load_data(DATA_DIR)
print(source)
report = run(transactions, articles, end, OUTPUT, synthetic=False,
             train_users=TRAIN_USERS, eval_users=EVAL_USERS, seed=SEED, source=source)


In [ ]:
import pandas as pd
from IPython.display import display, Markdown
display(pd.DataFrame.from_dict(report['map_at_12'], orient='index', columns=['MAP@12']))
display(Markdown((OUTPUT / 'report.md').read_text(encoding='utf-8')))
print('Paired bootstrap interval:', report['learned_minus_popular_95pct_bootstrap_ci'])
print('Training feature max:', report['training_feature_max_date'])
print('Evaluation feature max:', report['evaluation_feature_max_date'])


## Optional submission CSV

The full customer list is large; per-customer scoring may take considerable time.
Leave this disabled for a first evaluation. Creation does not upload anything.
Check current rules before sharing data-derived model or demo artifacts.


In [ ]:
CREATE_SUBMISSION = False
if CREATE_SUBMISSION:
    from fashion_recs.pipeline import submit
    submit(OUTPUT / 'model.pkl', DATA_DIR / 'sample_submission.csv', OUTPUT / 'submission.csv')
